# BETANIA — GPU gratis en Kaggle

Este cuaderno ejecuta **ComfyUI + Wan 2.2 TI2V-5B** dentro de una sesión GPU de Kaggle y usa el backend BETANIA del repositorio. No crea un servidor permanente ni un enlace público: la sesión es temporal y se controla desde este cuaderno.

**Antes de ejecutar:** en Kaggle abre Settings → Accelerator → GPU (P100 o T4 x2 si está disponible) y activa Internet. El modelo ocupa varios GB y la primera descarga puede tardar.

In [ ]:
import torch, os, subprocess, sys, time, json, requests, pathlib, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')
if not torch.cuda.is_available(): raise RuntimeError('Activa Accelerator → GPU en Kaggle y vuelve a ejecutar.')
!nvidia-smi

In [ ]:
# Instalar ComfyUI y el backend oficial de BETANIA
%cd /kaggle/working
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
!git clone --depth 1 https://github.com/caridad29932-cmd/generador-biblico-3d.git
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub
%cd /kaggle/working/generador-biblico-3d/backend
!pip install -q -r requirements.txt
print('Instalación terminada.')

In [ ]:
# Descargar los tres componentes oficiales del modelo Wan 2.2 TI2V-5B
from huggingface_hub import hf_hub_download
import shutil
root = pathlib.Path('/kaggle/working/ComfyUI/models')
files = [
 ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',root/'diffusion_models'),
 ('Comfy-Org/Wan_2.1_ComfyUI_repackaged','split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',root/'text_encoders'),
 ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/vae/wan2.2_vae.safetensors',root/'vae'),
]
for repo, filename, folder in files:
    folder.mkdir(parents=True, exist_ok=True)
    print('Descargando:', filename)
    cached = hf_hub_download(repo_id=repo, filename=filename, local_dir=str(folder))
    target = folder / pathlib.Path(filename).name
    if pathlib.Path(cached) != target:
        if target.exists(): target.unlink()
        shutil.move(cached, target)
    print('Guardado:', target)
print('Modelos descargados. Comprueba que Kaggle no haya agotado el espacio disponible.')

In [ ]:
# Iniciar ComfyUI y la API de BETANIA dentro de esta sesión
import secrets
BETANIA_KEY = secrets.token_urlsafe(32)
comfy_log = open('/kaggle/working/comfyui.log','w')
comfy = subprocess.Popen([sys.executable,'/kaggle/working/ComfyUI/main.py','--listen','127.0.0.1','--port','8188','--lowvram'], stdout=comfy_log, stderr=subprocess.STDOUT)
backend_log = open('/kaggle/working/betania-api.log','w')
env = os.environ.copy(); env['COMFYUI_URL']='http://127.0.0.1:8188'; env['BETANIA_API_KEY']=BETANIA_KEY; env['BETANIA_ORIGIN']='*'
api = subprocess.Popen([sys.executable,'-m','uvicorn','main:app','--host','127.0.0.1','--port','8000'], cwd='/kaggle/working/generador-biblico-3d/backend', env=env, stdout=backend_log, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        r=requests.get('http://127.0.0.1:8000/health',timeout=3)
        if r.ok and r.json().get('configured'): break
    except Exception: pass
    time.sleep(5)
print(requests.get('http://127.0.0.1:8000/health',timeout=10).json())
print('BETANIA API iniciada. Ejecuta la siguiente celda para crear la conexión temporal segura.')

In [ ]:
# Crear URL temporal para que la página BETANIA pueda comunicarse con esta sesión
# Cloudflare Quick Tunnel: URL temporal; se cierra al terminar o caducar la sesión Kaggle.
import re, time, subprocess, requests, os, pathlib
cloudflared='/kaggle/working/cloudflared'
if not pathlib.Path(cloudflared).exists():
    url='https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    data=requests.get(url,timeout=120); data.raise_for_status(); pathlib.Path(cloudflared).write_bytes(data.content); os.chmod(cloudflared,0o755)
tunnel_log=open('/kaggle/working/cloudflared.log','w')
tunnel=subprocess.Popen([cloudflared,'tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(90):
    time.sleep(2)
    try:
        log=pathlib.Path('/kaggle/working/cloudflared.log').read_text(errors='ignore')
        match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
        if match: public_url=match.group(0); break
    except Exception: pass
if not public_url: raise RuntimeError('No se obtuvo URL temporal. Revisa /kaggle/working/cloudflared.log')
print('\n=== COPIA ESTOS DATOS EN BETANIA ===')
print('DIRECCIÓN DEL SERVIDOR:',public_url)
print('CLAVE API:',BETANIA_KEY)
print('En BETANIA: pega la dirección en Servidor GPU y la clave en Clave API; pulsa Guardar y Probar conexión.')
print('No compartas la clave. La dirección cambia cuando reinicias Kaggle.')
print('Estado:',requests.get(public_url+'/health',timeout=30).json())

In [ ]:
# Escribe el prompt de UNA escena bíblica y genera el clip
prompt = input('Describe la escena bíblica (en español o inglés): ').strip()
if len(prompt) < 5: raise ValueError('Escribe una descripción más larga.')
job = requests.post('http://127.0.0.1:8000/api/jobs', headers={'Authorization':'Bearer '+BETANIA_KEY}, json={'prompt':prompt,'scene_index':1,'title':'BETANIA'}, timeout=180).json()
job_id = job['job_id']; print('Trabajo:', job_id)
for i in range(180):
    status = requests.get(f'http://127.0.0.1:8000/api/jobs/{job_id}', headers={'Authorization':'Bearer '+BETANIA_KEY}, timeout=30).json()
    print(status)
    if status.get('status') in ('completed','failed'): break
    time.sleep(20)
if status.get('status') != 'completed':
    print('No terminó dentro de este tiempo. Revisa /kaggle/working/comfyui.log')
else:
    video = requests.get('http://127.0.0.1:8000'+status['video_url'], timeout=600)
    video.raise_for_status()
    out='/kaggle/working/BETANIA_escena.mp4'
    open(out,'wb').write(video.content)
    print('VIDEO LISTO:',out, 'bytes:',len(video.content))
    from IPython.display import Video, display
    display(Video(out, embed=True))

## Importante
- Esta prueba genera un clip por ejecución; después puedes usar los botones de escenas en BETANIA mientras la sesión y el túnel sigan activos.
- Kaggle corta las sesiones y limita las horas GPU. No es alojamiento permanente.
- La URL Quick Tunnel es temporal; al reiniciar Kaggle debes ejecutar de nuevo las celdas y copiar la nueva dirección y clave.
- No publiques la clave API. El túnel expone el backend y la clave protege las solicitudes de generación.
- Si falla, revisa `/kaggle/working/comfyui.log`, `/kaggle/working/betania-api.log` y `/kaggle/working/cloudflared.log`.